# LedgerShield v2 - TRL SFT Training Notebook

This notebook demonstrates how to fine-tune a language model on LedgerShield v2 trajectories using TRL (Transformers Reinforcement Learning).

The notebook:
1. Sets up the environment with required dependencies
2. Loads the SFT training dataset from artifacts
3. Configures TRL for supervised fine-tuning
4. Runs training on a small model
5. Saves trained model and metrics

## Step 1: Install Dependencies

Install all required packages for TRL-based fine-tuning:

In [ ]:
# Install TRL and dependencies
!pip install -q transformers datasets accelerate peft trl torch bitsandbytes
print('✓ Dependencies installed')

## Step 2: Setup and Configuration

Configure paths, model selection, and training parameters:

In [ ]:
import json
from pathlib import Path
from typing import Any
import torch

# Configuration
config = {
    'model_name': 'Qwen/Qwen2.5-0.5B-Instruct',  # Small model for demo
    'input_file': 'artifacts/ledgershield_sft_examples.jsonl',
    'output_dir': 'artifacts/trl-sft-trained',
    'num_train_epochs': 1,
    'per_device_train_batch_size': 4,
    'learning_rate': 2e-5,
    'max_seq_length': 1024,
    'gradient_accumulation_steps': 1,
}

print('Configuration:')
for key, value in config.items():
    print(f'  {key}: {value}')

# Check if CUDA is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\n✓ Using device: {device}')

## Step 3: Load Training Data

Load the SFT examples generated from LedgerShield trajectories:

In [ ]:
def load_sft_examples(path: str) -> list[dict[str, Any]]:
    """Load JSONL training examples."""
    examples = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                examples.append(json.loads(line))
    return examples

# Load examples
examples = load_sft_examples(config['input_file'])
print(f'✓ Loaded {len(examples)} training examples')

# Show sample
if examples:
    sample = examples[0]
    print(f'\nSample example:')
    print(f'  Keys: {list(sample.keys())}')
    if 'prompt' in sample:
        prompt_preview = sample['prompt'][:100] + '...' if len(sample['prompt']) > 100 else sample['prompt']
        print(f'  Prompt: {prompt_preview}')
    if 'completion' in sample:
        completion_preview = sample['completion'][:100] + '...' if len(sample['completion']) > 100 else sample['completion']
        print(f'  Completion: {completion_preview}')

## Step 4: Prepare Dataset for TRL

Format examples for TRL's SFTTrainer:

In [ ]:
from datasets import Dataset

# Convert examples to HuggingFace Dataset format
def format_for_trl(examples: list[dict[str, Any]]) -> list[dict[str, str]]:
    """Format examples as text for TRL trainer."""
    formatted = []
    for ex in examples:
        # Combine prompt and completion as training text
        text = f"Prompt: {ex.get('prompt', '')}\nCompletion: {ex.get('completion', '')}"
        formatted.append({'text': text})
    return formatted

formatted_examples = format_for_trl(examples)
dataset = Dataset.from_dict({'text': [ex['text'] for ex in formatted_examples]})

print(f'✓ Dataset prepared with {len(dataset)} examples')
print(f'  Dataset format: {dataset.column_names}')
print(f'  Sample text length: {len(dataset[0]["text"])} chars')

## Step 5: Configure and Run TRL Training

Set up the SFTTrainer and run fine-tuning:

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer
from peft import LoraConfig

# Load model and tokenizer
print(f'Loading model: {config["model_name"]}')
tokenizer = AutoTokenizer.from_pretrained(config['model_name'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    config['model_name'],
    device_map=device,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
)
print(f'✓ Model loaded: {model.config.model_type}')

# Configure LoRA for efficient fine-tuning
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    bias='none',
    task_type='CAUSAL_LM',
)
print('✓ LoRA config created')

# Training arguments
training_args = TrainingArguments(
    output_dir=config['output_dir'],
    num_train_epochs=config['num_train_epochs'],
    per_device_train_batch_size=config['per_device_train_batch_size'],
    gradient_accumulation_steps=config['gradient_accumulation_steps'],
    learning_rate=config['learning_rate'],
    lr_scheduler_type='cosine',
    max_steps=len(dataset) // config['per_device_train_batch_size'],
    save_strategy='epoch',
    logging_steps=1,
    remove_unused_columns=True,
    optim='paged_adamw_8bit' if device == 'cuda' else 'adamw_torch',
)

# Create trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
    packing=False,
    max_seq_length=config['max_seq_length'],
    peft_config=lora_config,
)

print('✓ Trainer initialized')

## Step 6: Execute Training

Start the fine-tuning process:

In [ ]:
# Train
print('Starting training...')
train_result = trainer.train()
print(f'✓ Training complete')
print(f'  Final loss: {train_result.training_loss:.4f}')
print(f'  Training duration: {train_result.training_hours:.2f}h')

## Step 7: Save Model and Metrics

Save the fine-tuned model and training metrics:

In [ ]:
# Save model
output_path = Path(config['output_dir'])
output_path.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(output_path / 'final_model'))
tokenizer.save_pretrained(str(output_path / 'tokenizer'))

# Save metrics
metrics = {
    'model': config['model_name'],
    'training_examples': len(examples),
    'epochs': config['num_train_epochs'],
    'learning_rate': config['learning_rate'],
    'final_loss': float(train_result.training_loss),
    'training_duration_hours': train_result.training_hours,
    'status': 'completed',
}

metrics_path = output_path / 'training_metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'✓ Model saved to {output_path / "final_model"}')
print(f'✓ Metrics saved to {metrics_path}')
print(f'\nTraining Summary:')
for key, value in metrics.items():
    print(f'  {key}: {value}')

## Step 8: Quick Inference Test

Test the fine-tuned model with a quick inference:

In [ ]:
# Load fine-tuned model for inference
from peft import AutoPeftModelForCausalLM

ft_model = AutoPeftModelForCausalLM.from_pretrained(
    str(output_path / 'final_model'),
    device_map=device,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
)

# Test inference
test_prompt = examples[0].get('prompt', 'Test prompt') if examples else 'Test prompt'
test_input = tokenizer(test_prompt, return_tensors='pt').to(device)

with torch.no_grad():
    output = ft_model.generate(
        **test_input,
        max_new_tokens=100,
        temperature=0.7,
    )

generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print('✓ Inference test complete')
print(f'\nGenerated text (first 200 chars):')
print(generated_text[:200] + '...')

## Summary

Training complete! The fine-tuned model has been saved to `artifacts/trl-sft-trained/`.

To use the trained model:
```python
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

model = AutoPeftModelForCausalLM.from_pretrained('artifacts/trl-sft-trained/final_model')
tokenizer = AutoTokenizer.from_pretrained('artifacts/trl-sft-trained/tokenizer')
```

For more information on LedgerShield v2, see the main README.md